# Minute Lag code Guide

First you need to load in you **libraries**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema

Next you will declare your  **paths**. 

In [ ]:
PATH = '~/Library/Mobile Documents/com~apple~CloudDocs/Desktop/'
FILENAME_POS3_Mech = 'localizedvelocity_analyticsmCherryPOS3algo-start.csv'
FILENAME_POS3_Mito = 'wave_source_analyticsPOS3ratio-start.csv'

Then you will **pull your data** from your files. 

In [ ]:
dataMech3 = pd.read_csv(PATH + FILENAME_POS3_Mech, sep=',')
dataMito3 = pd.read_csv(PATH + FILENAME_POS3_Mito, sep=',')
TimeMech3 = dataMech3['Time']
ActivityMech3 = dataMech3['Avg_Speed']
TimeMito3 = dataMito3['Time(min)'] 
ActivityMito3 = dataMito3['GlobalWaveActivityMetric']

We once again are using a **Z-score** function to normalize our data. 

In [ ]:
def rolling_zscore3(series, window):
    rolling_mean = series.rolling(window=window, center=True, min_periods=1).mean()
    rolling_std = series.rolling(window=window, center=True, min_periods=1).std()
    return (series - rolling_mean) / rolling_std

Then we **apply this normalization**. 

In [1]:
window_frames = int(300 / 3.0) 
norm_ActivityMech3 = rolling_zscore3(ActivityMech3, window_frames)
norm_ActivityMito3 = rolling_zscore3(ActivityMito3, window_frames)

NameError: name 'rolling_zscore3' is not defined

Now, I included the code for a **fourier transform** but I did not use it in this analysis. I chose to not do it because I thought seeing the localized trends would be important. 

Hence why our threshold_percents are set to **0.0**.

In [ ]:
def fft_denoise3(signal3, threshold_percent3=.05):
    fft_vals3 = np.fft.fft(signal3)
    power3 = np.abs(fft_vals3) ** 2
    max_power3 = np.max(power3)
    threshold3 = threshold_percent3 * max_power3
    fft_vals_clean3 = fft_vals3.copy()
    fft_vals_clean3[power3 < threshold3] = 0
    return np.fft.ifft(fft_vals_clean3).real
clean_ActivityMech3 = fft_denoise3(norm_ActivityMech3.dropna(), threshold_percent3=0.0)
clean_ActivityMito3 = fft_denoise3(norm_ActivityMito3.dropna(), threshold_percent3=0.0)

Next, we continute to normalize the data but also identify the **troughs** and **peaks** of both of the data sets by finding the max and mins locally. 

In [ ]:
clean_TimeMech3 = TimeMech3[norm_ActivityMech3.notna()].reset_index(drop=True)
clean_TimeMito3 = TimeMito3[norm_ActivityMito3.notna()].reset_index(drop=True)

max_indMECH3 = argrelextrema(clean_ActivityMech3, np.greater)[0]
max_indMITO3 = argrelextrema(clean_ActivityMito3, np.greater)[0]

Mechmaxpts3 = clean_ActivityMech3[max_indMECH3]
Mitomaxpts3 = clean_ActivityMito3[max_indMITO3]

trough_indMECH3 = argrelextrema(clean_ActivityMech3, np.less)[0]
trough_indMITO3 = argrelextrema(clean_ActivityMito3, np.less)[0]
mito_trough_times3 = clean_TimeMito3.iloc[trough_indMITO3].values

Lets **plot** this data.

In [ ]:
plt.figure(figsize=(20, 5))
plt.title('Denoised RFP wave speeds vs. Mitotic waves POS 3 - compartment forming sample', fontsize=14, fontweight='bold')
plt.plot(clean_TimeMech3, clean_ActivityMech3, linestyle="-", color="red", linewidth=2, label='mCherry (FFT Clean)')
plt.plot(clean_TimeMito3, clean_ActivityMito3, linestyle="-", color="green", linewidth=2, label='FRET/CFP (FFT Clean')
plt.xlabel('Time (minutes)')
plt.xlim(TimeMech3.min(), TimeMech3.max())
plt.legend(loc="upper right")
for x in range(0,240,10):
    plt.axvline(x, linestyle='--', alpha=0.3, color='black') 
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

Next, we will **identify the peaks** in both of the data sets and **determine their relative lags**. 

In [ ]:
#phase lag stuff 
mech_peak_times = clean_TimeMech3.iloc[max_indMECH3].values 
mito_peak_times = clean_TimeMito3.iloc[max_indMITO3].values 

phase_delay = []
time_midpoint = []

for mech_time in mech_peak_times:
    nearest = mito_peak_times[np.argmin(np.abs(mito_peak_times - mech_time))]
    delta_t = nearest - mech_time
    idx = np.where(mech_peak_times == mech_time)[0][0]

    if idx < len(mech_peak_times)-1:
        period = mech_peak_times[idx+1] - mech_peak_times[idx]
    else:
        period = np.mean(np.diff(mech_peak_times))
        
    phase = (2*np.pi*delta_t)/period
    phase_delay.append(phase)
    time_midpoint.append(mech_time)
phase_delay = np.array(phase_delay)

We can then **print out these results**. 

In [ ]:
print("\nExperimental Phase Lag\n")
for t,p in zip(time_midpoint, phase_delay):
    print(f"Time = {t:6.1f} min    Phase Lag = {p:6.3f} rad")

This code is not perfect so you will need to make sense of some of you results. In this case you might choose to apply the **fourier transform** to the data if it is too noisy. 

To visualize these results, we can **plot how the lag changes over time**. 

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(time_midpoint,phase_delay,'o-',linewidth=2)
plt.axhline(0,color='k',linestyle='--')
plt.xlabel("Time (minutes)")
plt.ylabel("Relative Phase Lag (radians)")
plt.title("Experimental Relative Phase Lag")
plt.grid(True)
plt.show()

# End of Walkthrough

## Full Code Below: 

In [ ]:
# POS 3 - cycle 7

#library calls
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema

#determining the paths
PATH = '~/Library/Mobile Documents/com~apple~CloudDocs/Desktop/'
FILENAME_POS3_Mech = 'localizedvelocity_analyticsmCherryPOS3algo-test8.csv'
FILENAME_POS3_Mito = 'wave_source_analyticsPOS3ratio-test8.csv'

#set our data 
dataMech3 = pd.read_csv(PATH + FILENAME_POS3_Mech, sep=',')
dataMito3 = pd.read_csv(PATH + FILENAME_POS3_Mito, sep=',')

TimeMech3 = dataMech3['Time']
ActivityMech3 = dataMech3['Avg_Speed']
TimeMito3 = dataMito3['Time(min)']
ActivityMito3 = dataMito3['GlobalWaveActivityMetric']

#z-score function
def rolling_zscore3(series, window):
    rolling_mean = series.rolling(window=window, center=True, min_periods=1).mean()
    rolling_std = series.rolling(window=window, center=True, min_periods=1).std()
    return (series - rolling_mean) / rolling_std

window_frames = int(300 / 3.0) 
norm_ActivityMech3 = rolling_zscore3(ActivityMech3, window_frames)
norm_ActivityMito3 = rolling_zscore3(ActivityMito3, window_frames)

#fourier transform 
def fft_denoise3(signal3, threshold_percent3=.05):
    fft_vals3 = np.fft.fft(signal3)
    power3 = np.abs(fft_vals3) ** 2
    max_power3 = np.max(power3)
    threshold3 = threshold_percent3 * max_power3
    fft_vals_clean3 = fft_vals3.copy()
    fft_vals_clean3[power3 < threshold3] = 0
    return np.fft.ifft(fft_vals_clean3).real

clean_ActivityMech3 = fft_denoise3(norm_ActivityMech3.dropna(), threshold_percent3=0.1)
clean_ActivityMito3 = fft_denoise3(norm_ActivityMito3.dropna(), threshold_percent3=0.0)

clean_TimeMech3 = TimeMech3[norm_ActivityMech3.notna()].reset_index(drop=True)
clean_TimeMito3 = TimeMito3[norm_ActivityMito3.notna()].reset_index(drop=True)

#finding local peaks
max_indMECH3 = argrelextrema(clean_ActivityMech3, np.greater)[0]
max_indMITO3 = argrelextrema(clean_ActivityMito3, np.greater)[0]

Mechmaxpts3 = clean_ActivityMech3[max_indMECH3]
Mitomaxpts3 = clean_ActivityMito3[max_indMITO3]

mech_peak_times = clean_TimeMech3.iloc[max_indMECH3].values 
mito_peak_times = clean_TimeMito3.iloc[max_indMITO3].values 

#finding local troughs
trough_indMECH3 = argrelextrema(clean_ActivityMech3, np.less)[0]
trough_indMITO3 = argrelextrema(clean_ActivityMito3, np.less)[0]
mito_trough_times3 = clean_TimeMito3.iloc[trough_indMITO3].values

#printing the peak summary
#this is helpful to discern your results!!!!!!!

print("=" * 45)
print("             MECHANICAL PEAKS (mCherry)")
print("=" * 45)
for t, val in zip(mech_peak_times, Mechmaxpts3):
    print(f"Time: {t:6.2f} min   |   Amplitude: {val:6.3f}")

print("\n" + "=" * 45)
print("             MITOTIC PEAKS (FRET/CFP)")
print("=" * 45)
for t, val in zip(mito_peak_times, Mitomaxpts3):
    print(f"Time: {t:6.2f} min   |   Amplitude: {val:6.3f}")
print("=" * 45)


#plotting our data 
plt.figure(figsize=(20, 5))
plt.title('Denoised RFP wave speeds vs. Mitotic waves POS 3 - compartment forming sample', fontsize=14, fontweight='bold')
plt.plot(clean_TimeMech3, clean_ActivityMech3, linestyle="-", color="red", linewidth=2, label='mCherry (FFT Clean)')
plt.plot(clean_TimeMito3, clean_ActivityMito3, linestyle="-", color="green", linewidth=2, label='FRET/CFP (FFT Clean)')
plt.scatter(mech_peak_times, Mechmaxpts3, color='darkred', s=60, zorder=5, label='Mech Peaks')
plt.scatter(mito_peak_times, Mitomaxpts3, color='darkgreen', s=60, zorder=5, label='Mito Peaks')
plt.xlabel('Time (minutes)')
plt.xlim(TimeMech3.min(), TimeMech3.max())
plt.legend(loc="upper right")
for x in range(46, 240, 10): # you are going to have to change this depending on your data, play around with it!
    plt.axvline(x, linestyle='--', alpha=0.3, color='black') 
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

#finding the lag 
phase_delay = []
time_midpoint = []

for mech_time in mech_peak_times:
    nearest = mito_peak_times[np.argmin(np.abs(mito_peak_times - mech_time))]
    delta_t = nearest - mech_time
    idx = np.where(mech_peak_times == mech_time)[0][0]
    if idx < len(mech_peak_times) - 1:
        period = mech_peak_times[idx + 1] - mech_peak_times[idx]
    else:
        period = np.mean(np.diff(mech_peak_times)) 
    phase = (2 * np.pi * delta_t) / period
    phase_delay.append(phase)
    time_midpoint.append(mech_time)
phase_delay = np.array(phase_delay)

#print out our results
print("\nExperimental Phase Lag\n")
for t, p in zip(time_midpoint, phase_delay):
    print(f"Time = {t:6.1f} min    Phase Lag = {p:6.3f} rad")

#plotting our results
plt.figure(figsize=(10, 4))
plt.plot(time_midpoint, phase_delay, 'o-', linewidth=2)
plt.axhline(0, color='k', linestyle='--')
plt.xlabel("Time (minutes)")
plt.ylabel("Relative Phase Lag (radians)")
plt.title("Experimental Relative Phase Lag")
plt.grid(True)
plt.tight_layout()
plt.show()